In [ ]:
posts_path =r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts.csv"
comments_path = r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts_comments.csv"

In [ ]:
import pandas as pd
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import torch
from collections import Counter


In [ ]:
post_df = pd.read_csv(posts_path)
comment_df = pd.read_csv(comments_path, sep = ';')

In [ ]:
comment_agg = comment_df.groupby('post_id')['comments'].apply(list).reset_index()
merged_df = post_df.merge(comment_agg, on='post_id', how='left')

In [ ]:
df = merged_df.drop(columns=['image'])
df.head()

In [ ]:
print ("Rows : " ,df.shape[0])
print ("Columns : " ,df.shape[1])
print ("\nFeatures : \n" ,df.columns.tolist())

In [ ]:
missing_values = df.isnull().sum()
print(missing_values)

In [ ]:
# Fill missing value in caption and hashtags column by ""
df['caption'] = df['caption'].fillna('')
df['hashtags'] = df['hashtags'].fillna('')

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

### "Caption" Column

In [ ]:
# Clean caption
import re

def clean_caption(text):
    if pd.isna(text) or text == 'NaN':
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^\w\s\U0001F000-\U0001F9FF]', ' ', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['clean_caption'] = df['caption'].apply(clean_caption)

In [50]:
# Process Emoji in caption
import emoji

def process_emoji(text):
    emojis_list = [c for c in text if c in emoji.EMOJI_DATA]
    
    emoji_descriptions = []
    for e in emojis_list:
        emoji_name = emoji.demojize(e).replace(':', '').replace('_', '   ')
        emoji_descriptions.append(emoji_name)
    
    text_without_emoji = ''.join(c for c in text if c not in emoji.EMOJI_DATA)
    
    return text_without_emoji, emoji_descriptions

df[['caption_no_emoji', 'emoji_descriptions']] = df['clean_caption'].apply(lambda x: pd.Series(process_emoji(x)))


### "Hashtag" column

In [ ]:
# Process hashtags from hashtag column
def process_hashtags_from_column(hashtag_text):
    if pd.isna(hashtag_text) or hashtag_text == 'nan' or hashtag_text == '':
        return []
    
    if hashtag_text.startswith('[') and hashtag_text.endswith(']'):
        hashtag_text = hashtag_text[1:-1]
        
        tags = [tag.strip().strip("'").strip('"') for tag in hashtag_text.split(',')]
    else:
        tags = [tag.strip().strip('#') for tag in re.split(r'[,#]', hashtag_text) if tag.strip()]
    
    processed_hashtags = []
    for tag in tags:
        if tag:
            words = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', tag)
            if not words:
                words = [tag]
            processed_hashtags.extend([word.lower() for word in words])
    
    return processed_hashtags

df['hashtags'] = df['hashtags'].apply(process_hashtags_from_column)

In [ ]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('stopwords')


# Tokenize and remove stopwords
def tokenize_and_remove_stopwords(text):
    
    tokens = word_tokenize(text)
    
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    
    return filtered_tokens

df['caption_tokens'] = df['caption_no_emoji'].apply(tokenize_and_remove_stopwords)
df['combined_tokens'] = df.apply(lambda row: row['caption_tokens'] + row['emoji_descriptions'] + row['hashtags'], axis=1)

In [ ]:
df.head()

### BERT Embedding for combined_tokens

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

In [ ]:
def get_bert_embedding(tokens):
    """Convert combined_tokens (list of words) to a BERT embedding."""
    text = " ".join(tokens)  # Convert list to a single sentence
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()  # Take mean embedding

# Apply to dataframe
df['bert_embedding'] = df['combined_tokens'].apply(get_bert_embedding)

print(df[['combined_tokens', 'bert_embedding']].head())